# Week 4: Machine Learning for Forecasting
## In-Class Exercises

**Objective.** Turn a time series into a supervised learning problem without fooling yourself, fit gradient boosting to it, and find out where trees beat ETS and where they fall over.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | `mlforecast` end to end, plus the failure that motivates everything after it. |
| 2 | Blanks we fill in together | Dummies vs. Fourier terms for weekly seasonality. |
| 3 | On your own, ~15 min | Build the lag matrix by hand, then recursive vs. direct forecasting. |

In [ ]:
!pip install -q mlforecast lightgbm scikit-learn pandas plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.linear_model import LinearRegression, Ridge
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import Differences

pio.templates.default = "plotly_white"   # try "plotly_dark", "ggplot2", "simple_white"

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = (pd.read_csv(URL, parse_dates=["Month"])
         .rename(columns={"Month": "ds", "Passengers": "y"})
         .assign(unique_id="airline"))

h, m = 24, 12
train = air.iloc[:-h].copy()
test = air.iloc[-h:].copy()


def mae(y, yhat):
    return np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float)))

def mase(y, yhat, y_train, m=12):
    y_train = np.asarray(y_train, float)
    return mae(y, yhat) / np.mean(np.abs(y_train[m:] - y_train[:-m]))


def forecast_plot(preds, title, history=train, tail=None):
    """History plus test actuals plus one line per model column in `preds`."""
    hist = history.tail(tail) if tail else history
    tidy = pd.concat([
        hist.assign(series="train").rename(columns={"y": "value"})[["ds", "series", "value"]],
        test.assign(series="test").rename(columns={"y": "value"})[["ds", "series", "value"]],
        preds.melt(id_vars="ds", value_vars=[c for c in preds if c not in ("unique_id", "ds")],
                   var_name="series", value_name="value"),
    ])
    return px.line(tidy, x="ds", y="value", color="series", title=title)


train.tail(3)

---
## Part 1. The ML framing, and the trap inside it

A forecasting model becomes a regression model the moment you write

$$y_t = f(y_{t-1}, y_{t-2}, \ldots, y_{t-m}, \text{calendar features}) + \varepsilon_t$$

`mlforecast` builds that design matrix, fits any scikit-learn-style model to it, and rolls the predictions forward recursively.

Watch the first fit. It fails in a specific, instructive way.

In [ ]:
fcst = MLForecast(
    models={"lr": LinearRegression(), "gbm": LGBMRegressor(n_estimators=200, verbosity=-1)},
    freq="MS",
    lags=[1, 12],
    lag_transforms={1: [RollingMean(window_size=12)]},
    date_features=["month"],
)
fcst.fit(train)
p_raw = fcst.predict(h)

forecast_plot(p_raw, "First attempt: no target transform").show()

for mdl in ["lr", "gbm"]:
    print(mdl, "MASE:", round(mase(test["y"], p_raw[mdl], train["y"], m), 3))

**The failure, and why it matters.**

The gradient boosting forecast flattens and never exceeds the highest value it saw in training. Not a bug, not a tuning problem. **A tree predicts an average of training targets, so it cannot produce a number outside the training range.** On a trending series that is fatal, and no amount of `n_estimators` fixes it.

Linear regression extrapolates, which is why the plain `lr` line often beats the boosted one here.

The fix: remove the trend before the model sees the data and add it back afterward. That is `Differences([1])`.

In [ ]:
fcst2 = MLForecast(
    models={"lr": LinearRegression(), "gbm": LGBMRegressor(n_estimators=200, verbosity=-1)},
    freq="MS",
    lags=[1, 12],
    lag_transforms={1: [RollingMean(window_size=12)]},
    date_features=["month"],
    target_transforms=[Differences([1])],     # <- the whole difference
)
fcst2.fit(train)
p = fcst2.predict(h)

forecast_plot(p, "With Differences([1])", tail=60).show()

snaive = np.tile(train["y"].to_numpy()[-m:], int(np.ceil(h / m)))[:h]
print("seasonal naive MASE:", round(mase(test["y"], snaive, train["y"], m), 3))
for mdl in ["lr", "gbm"]:
    print(f"{mdl} MASE:", round(mase(test["y"], p[mdl], train["y"], m), 3))

**Notice the benchmark line.** With 120 observations and one series, seasonal naive is a serious competitor and gradient boosting may still lose to it. ML methods earn their keep on *panels*: hundreds or thousands of related series pooled into one global model. That is Week 5. Today is mechanics and traps.

---
## Part 2. Dummies vs. Fourier for seasonality

Two ways to encode weekly seasonality on daily data:

- **Dummies**: 6 binary columns (7 days minus a reference). Fully flexible, 6 parameters.
- **Fourier terms**: $\sin$ and $\cos$ pairs at period 7. Smooth, $2K$ parameters for $K$ pairs.

At period 7 there is no strong reason to prefer either. At period 365.25, dummies need 364 columns and Fourier needs about 6. That is the real trade-off.

The series below has weekly *and* yearly seasonality, so both regimes are visible at once.

In [ ]:
rng = np.random.default_rng(11)
n_days = 365 * 3
idx = pd.date_range("2021-01-01", periods=n_days, freq="D")
t = np.arange(n_days)

y = (200
     + 0.05 * t
     + 15 * np.sin(2 * np.pi * t / 365.25)                 # yearly
     + np.where(idx.dayofweek >= 5, -25, 8)                # weekend effect
     + rng.normal(scale=6, size=n_days))
daily = pd.Series(y, index=idx, name="y")

px.line(daily.iloc[:120].rename_axis("date").reset_index(), x="date", y="y",
        title="Simulated daily series, first four months").show()

In [ ]:
def fourier_terms(index, period, K, prefix):
    """K sin/cos pairs at the given period, as a DataFrame aligned to index."""
    t = np.arange(len(index))
    cols = {}
    for k in range(1, K + 1):
        cols[f"{prefix}_sin{k}"] = np.sin(2 * np.pi * k * t / period)
        cols[f"{prefix}_cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=index)


H = 90
tr_idx, te_idx = daily.index[:-H], daily.index[-H:]

# Design A: day-of-week dummies + yearly Fourier
dow = pd.get_dummies(daily.index.dayofweek, prefix="dow", drop_first=True).set_index(daily.index).astype(float)
yearly = fourier_terms(daily.index, 365.25, K=3, prefix="yr")
X_dummy = pd.concat([dow, yearly, pd.Series(np.arange(n_days), index=daily.index, name="trend")], axis=1)

# Design B: weekly Fourier + yearly Fourier
# TODO - build X_fourier the same way, but replace `dow` with 3 Fourier pairs at period 7.
#   Hint: fourier_terms(daily.index, 7, K=3, prefix="wk")
X_fourier = ...

<details>
<summary><b>Show the line</b></summary>

```python
X_fourier = pd.concat([fourier_terms(daily.index, 7, K=3, prefix="wk"),
                       yearly,
                       pd.Series(np.arange(n_days), index=daily.index, name="trend")], axis=1)
```
</details>

In [ ]:
def fit_score(X, label):
    mdl = LinearRegression().fit(X.loc[tr_idx], daily.loc[tr_idx])
    pred = mdl.predict(X.loc[te_idx])
    resid = daily.loc[tr_idx] - mdl.predict(X.loc[tr_idx])
    n, k = len(tr_idx), X.shape[1] + 1
    aic = n * np.log(np.sum(resid ** 2) / n) + 2 * k
    aicc = aic + 2 * k * (k + 1) / (n - k - 1)
    return {"model": label, "n_features": X.shape[1], "AICc": round(aicc, 1),
            "test MAE": round(mae(daily.loc[te_idx], pred), 3)}


pd.DataFrame([fit_score(X_dummy, "dummies + yearly Fourier"),
              fit_score(X_fourier, "weekly Fourier + yearly Fourier")])

**Reading the table.** The two designs land within a hair of each other on test MAE, because 3 Fourier pairs at period 7 span almost the same space as 6 dummies. The interesting column is `n_features`. Encode yearly seasonality as day-of-year dummies and you get 364 columns on 3 years of data, roughly 3 observations per parameter. Fourier is not a better model, it is a cheaper one, and cheap is what makes long periods workable.

---
## Part 3. Building the lag matrix yourself

About 15 minutes. `mlforecast` hid the machinery; now you write it, because the bugs live in there.

**Tasks.**

1. Finish `make_supervised(y, lags)` so it returns `X` (one column per lag) and `y`, aligned, with no NaN rows.
2. Fit `Ridge` and `LGBMRegressor` on the airline training data with lags 1, 2, 3, 12, 13.
3. Forecast 24 steps **recursively**: predict one step, append it to the history, recompute lags, repeat.
4. Forecast 24 steps **directly**: one model per horizon $k$, target shifted $k$ steps ahead, using only lags available at forecast time.
5. Compare both to seasonal naive with MASE, and say which framing wins here and what would flip it.

Watch the alignment in step 1. An off-by-one there is the most common bug in this course, and it shows up as suspiciously excellent accuracy.

In [ ]:
def make_supervised(y, lags):
    """y: 1-d array. Returns (X, target) with rows containing no NaNs."""
    s = pd.Series(np.asarray(y, float))
    X = pd.DataFrame({f"lag{L}": s.shift(L) for L in lags})
    # TODO - align the target with X, then drop rows with any NaN
    target = ...
    keep = ...
    return X[keep], target[keep]


# Quick self-check once you have it: with lags=[1], X["lag1"] should equal target shifted by one.
# make_supervised(np.arange(10), [1])

In [ ]:
# YOUR CODE HERE - tasks 2 through 5

<details>
<summary><b>Solution</b></summary>

```python
def make_supervised(y, lags):
    s = pd.Series(np.asarray(y, float))
    X = pd.DataFrame({f"lag{L}": s.shift(L) for L in lags})
    target = s
    keep = X.notna().all(axis=1)
    return X[keep], target[keep]


LAGS = [1, 2, 3, 12, 13]
ytr = train["y"].to_numpy()
X, tgt = make_supervised(ytr, LAGS)

models = {"ridge": Ridge().fit(X, tgt),
          "gbm": LGBMRegressor(n_estimators=300, verbosity=-1).fit(X, tgt)}

#  recursive 
rec = {}
for name, mdl in models.items():
    hist = list(ytr)
    preds = []
    for _ in range(h):
        row = [[hist[-L] for L in LAGS]]
        nxt = float(mdl.predict(row)[0])
        preds.append(nxt)
        hist.append(nxt)
    rec[name] = np.array(preds)

#  direct: one model per horizon k 
# Row i of Xk holds lags measured k-1 steps before the target at row i,
# so at the forecast origin the feature row is simply the last observed lags.
from sklearn.base import clone

dir_preds = {name: np.empty(h) for name in models}
last_row = [[ytr[-L] for L in LAGS]]
for k in range(1, h + 1):
    s = pd.Series(ytr)
    Xk = pd.DataFrame({f"lag{L}": s.shift(L + k - 1) for L in LAGS})
    keep = Xk.notna().all(axis=1)
    Xk, tk = Xk[keep], s[keep]
    for name, base in models.items():
        dir_preds[name][k - 1] = clone(base).fit(Xk, tk).predict(last_row)[0]

snaive = np.tile(ytr[-m:], int(np.ceil(h / m)))[:h]
print("seasonal naive:", round(mase(test["y"], snaive, ytr, m), 3))
for name in models:
    print(f"{name} recursive:", round(mase(test["y"], rec[name], ytr, m), 3),
          " direct:", round(mase(test["y"], dir_preds[name], ytr, m), 3))
```

**Takeaways.**

- **Recursive** trains one model and feeds its own predictions back in, so errors compound with horizon. Efficient, and `mlforecast`'s default.
- **Direct** trains $h$ models, so nothing compounds, but each sees fewer rows and they are not constrained to be consistent with each other.
- Recursive usually wins at short horizons, direct often wins at long horizons on noisy series. No universal answer, which is why the comparison is the exercise.
- Trees still cannot extrapolate. If MASE is poor, difference the series and re-run. Same lesson as Part 1, arrived at the hard way.
</details>

---
## Wrap-up

1. **Trees cannot extrapolate.** Always difference or detrend a trending target.
2. **Recursive vs. direct is a real choice**, not an implementation detail.
3. **Time-aware CV or nothing.** Shuffled folds and centered rolling windows both leak the future.
4. **One short series is not where ML wins.** Next week: a panel, the setting these methods were built for.